# Q6 dev evidence + Q7 final test files

Q6 is not skipped: its dev experiment results already exist. Its final results table also needs the test score returned by Gradescope after Q7 submission. This notebook first exports the saved epoch 17 dev predictions and actual error examples for Q6, then generates the two Q7 files using the already selected model. No test-driven model selection, retraining, or automatic Gradescope submission occurs.

Use the **same current L4 runtime** where continuation training finished. Run Section **1**, then Section **2 once**. No uploads or package reinstalls are needed. The fixed checkpoint is **best epoch 17**, not the last epoch 20 model. Generation settings stay greedy, 512 maximum new tokens. The 432 test inputs are processed in their original order, and `<` markers are restored before SQL execution. SQL runs in the bounded worker used during dev evaluation.

New output lives under `MyDrive/NYU_NLP_HW4/part2/final_exports/<timestamp>/`. The submission ZIP contains exactly:
- `results/t5_ft_experiment_test.sql`
- `records/t5_ft_experiment_test.pkl`

Q6 evidence is exported separately under `q6_dev_analysis/`. Its automatically grouped execution errors have measured counts; semantic review hints are not finished error classifications. Review actual examples before writing at least three error classes in the report. Test F1 is **unknown until Gradescope returns it**; dev F1 is not a substitute.

Final message: **`Q6 EVIDENCE AND Q7 TEST FILES READY`**. Save this notebook and ask for verification before submitting the files yourself. Do not rerun generation just to inspect existing results; use Section 3 instead. Original homework source and model backups are not changed.


## 1. Prepare — no generation yet

Run once to locate and lock the completed epoch 17 model. A completed prior export blocks accidental regeneration.


In [1]:
FINAL_EXPORT_SOURCE = "\"\"\"Export Q6 dev evidence, then generate Q7 test predictions ONCE from the fixed best model.\"\"\"\nimport csv\nfrom collections import Counter, defaultdict\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport pickle\nimport re\nimport sys\nimport traceback\nfrom types import SimpleNamespace\nimport zipfile\n\n\ndef sha256(path):\n    value = hashlib.sha256()\n    with Path(path).open(\"rb\") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            value.update(block)\n    return value.hexdigest()\n\n\ndef execution_group(message):\n    lower = message.lower()\n    if not lower:\n        return None\n    if \"timed out\" in lower:\n        return \"SQL execution timeout\"\n    if any(s in lower for s in [\"syntax error\", \"incomplete input\", \"unrecognized token\"]):\n        return \"SQL syntax/format error\"\n    if any(s in lower for s in [\"no such column\", \"no such table\", \"ambiguous column\"]):\n        return \"SQL identifier/reference error\"\n    if any(s in lower for s in [\"misuse of aggregate\", \"wrong number of arguments\", \"no such function\"]):\n        return \"SQL function/aggregation error\"\n    return \"Other SQL execution error\"\n\n\ndef export_dev_evidence(context, output, utils):\n    work = Path(context[\"work_directory\"])\n    run = Path(context[\"training_run\"])\n    gold_sql = (work / \"data/dev.sql\").read_text().splitlines()\n    nl = (work / \"data/dev.nl\").read_text().splitlines()\n    predicted_path, predicted_records = run / \"predictions/best/dev.sql\", run / \"predictions/best/dev.pkl\"\n    predictions = predicted_path.read_text().splitlines()\n    with (work / \"records/ground_truth_dev.pkl\").open(\"rb\") as handle:\n        gold_records, gold_errors = pickle.load(handle)\n    with predicted_records.open(\"rb\") as handle:\n        records, errors = pickle.load(handle)\n    assert all(len(x) == 466 for x in [gold_sql, nl, predictions, gold_records, gold_errors, records, errors])\n    sql_em, record_em, f1, _ = utils.compute_metrics(\n        str(work / \"data/dev.sql\"), str(predicted_path),\n        str(work / \"records/ground_truth_dev.pkl\"), str(predicted_records),\n    )\n    expected = context[\"training_report\"]\n    for actual, name in [(sql_em, \"sql_em\"), (record_em, \"record_em\"), (f1, \"record_f1\")]:\n        assert math.isclose(float(actual), expected[name], abs_tol=1e-8)\n    rows, groups = [], defaultdict(list)\n    literal = re.compile(r\"'(?:''|[^'])*'|\\b\\d+(?:\\.\\d+)?\\b\")\n    for index, (question, gold, predicted, gold_rows, pred_rows, error) in enumerate(\n        zip(nl, gold_sql, predictions, gold_records, records, errors), 1,\n    ):\n        exact_records = set(gold_rows) == set(pred_rows)\n        group = execution_group(error)\n        row = {\"example_number_1_based\": index, \"nl\": question, \"gold_sql\": gold,\n               \"predicted_sql\": predicted, \"sql_exact_match\": gold.strip() == predicted.strip(),\n               \"record_exact_match\": exact_records,\n               \"record_f1\": float(utils.compute_record_F1([gold_rows], [pred_rows])),\n               \"sqlite_error\": error, \"execution_error_group\": group,\n               \"gold_record_count\": len(gold_rows), \"predicted_record_count\": len(pred_rows),\n               \"literal_sets_differ_review_hint_only\": sorted(literal.findall(gold)) != sorted(literal.findall(predicted))}\n        rows.append(row)\n        if group:\n            groups[group].append(row)\n        elif not exact_records:\n            groups[\"Valid SQL but incorrect records: manual semantic review needed\"].append(row)\n    summary = {\"model_epoch\": expected[\"best_epoch\"], \"dev_examples\": 466,\n               \"record_f1\": float(f1), \"record_em\": float(record_em), \"sql_em\": float(sql_em),\n               \"sql_execution_errors\": sum(bool(e) for e in errors),\n               \"record_mismatches\": sum(not r[\"record_exact_match\"] for r in rows),\n               \"sql_text_mismatches\": sum(not r[\"sql_exact_match\"] for r in rows),\n               \"review_group_counts_out_of_466\": {key: len(value) for key, value in groups.items()},\n               \"note\": \"Execution groups are measured from saved error messages. Semantic hints are NOT final error classifications; inspect examples before writing Q6.\"}\n    q6 = output / \"q6_dev_analysis\"\n    q6.mkdir()\n    (q6 / \"summary.json\").write_text(json.dumps(summary, indent=2))\n    (q6 / \"all_dev_examples.json\").write_text(json.dumps(rows, indent=2))\n    with (q6 / \"all_dev_examples.csv\").open(\"w\", newline=\"\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    samples = []\n    for group, examples in groups.items():\n        if group.startswith(\"Valid SQL\"):\n            differing = [r for r in examples if r[\"literal_sets_differ_review_hint_only\"]]\n            same = [r for r in examples if not r[\"literal_sets_differ_review_hint_only\"]]\n            chosen = differing[:3] + same[:3]\n        else:\n            chosen = examples[:2]\n        samples.extend({\"review_group\": group, **r} for r in chosen)\n    (q6 / \"review_examples.json\").write_text(json.dumps(samples, indent=2))\n    print(\"Q6 DEV EVIDENCE EXPORTED\", flush=True)\n    print(json.dumps(summary, indent=2), flush=True)\n    print(\"Q6 EXAMPLES FOR REVIEW (not a finished written answer):\", flush=True)\n    print(json.dumps(samples, indent=2), flush=True)\n    return summary\n\n\ndef main(context_path):\n    context = json.loads(Path(context_path).read_text())\n    work, output = Path(context[\"work_directory\"]), Path(context[\"export_directory\"])\n    assert Path.cwd().resolve() == work.resolve()\n    sys.path.insert(0, str(work))\n    import torch\n    from transformers import T5TokenizerFast\n    import utils\n    from load_data import get_dataloader\n    from t5_utils import load_model_from_checkpoint\n    from train_t5 import test_inference\n    from final_bounded_sql import install_bounded_sql\n    install_bounded_sql(utils)\n\n    def status(state, stage, **extra):\n        (output / \"status.json\").write_text(json.dumps({\n            \"state\": state, \"stage\": stage, \"updated_utc\": datetime.now(timezone.utc).isoformat(),\n            \"best_epoch\": context[\"training_report\"][\"best_epoch\"],\n            \"export_directory\": str(output), \"training_updates\": 0, **extra,\n        }, indent=2))\n\n    try:\n        status(\"RUNNING\", \"exporting_q6_dev_evidence\")\n        dev_summary = export_dev_evidence(context, output, utils)\n        args = SimpleNamespace(**context[\"configuration\"])\n        assert torch.cuda.is_available() and args.finetune\n        assert args.num_beams == 1 and args.max_new_tokens == 512\n        utils.set_random_seeds(args.seed)\n        checkpoint = Path(context[\"best_checkpoint_directory\"])\n        tokenizer = T5TokenizerFast.from_pretrained(str(checkpoint), local_files_only=True)\n        status(\"RUNNING\", \"loading_fixed_best_model_for_final_test\")\n        model = load_model_from_checkpoint(args, best=True)\n        assert next(model.parameters()).device.type == \"cuda\"\n        signature = model.shared.weight[:16, :16].detach().clone()\n        test_loader = get_dataloader(args.test_batch_size, \"test\")\n        raw_test = (work / \"data/test.nl\").read_text().splitlines()\n        assert len(raw_test) == len(test_loader.dataset) == 432\n        assert test_loader.dataset.target_ids is None\n        assert tokenizer(raw_test, padding=False, truncation=False)[\"input_ids\"] == test_loader.dataset.input_ids\n        assert tokenizer.get_vocab() == test_loader.dataset.tokenizer.get_vocab()\n        sql_path = output / \"results/t5_ft_experiment_test.sql\"\n        records_path = output / \"records/t5_ft_experiment_test.pkl\"\n        print(f\"\\nFINAL TEST GENERATION: fixed best epoch {context['training_report']['best_epoch']}, 432 examples in original order.\", flush=True)\n        print(\"No training or model selection on test. Test F1 is available only from Gradescope.\", flush=True)\n        status(\"RUNNING\", \"generating_test_sql_and_records\", test_examples=432)\n        test_inference(args, model, test_loader, str(sql_path), str(records_path))\n        queries = sql_path.read_text().splitlines()\n        with records_path.open(\"rb\") as handle:\n            records, errors = pickle.load(handle)\n        assert len(queries) == len(records) == len(errors) == 432\n        assert all(q.strip() for q in queries), \"Blank prediction: preserve files and ask for review.\"\n        assert all(\"less_than\" not in q for q in queries), \"Unrestored SQL marker.\"\n        assert all(isinstance(e, str) for e in errors)\n        assert not model.training and all(p.grad is None for p in model.parameters())\n        assert torch.equal(signature, model.shared.weight[:16, :16].detach())\n        bundle = output / \"Q7_submission_files.zip\"\n        with zipfile.ZipFile(bundle, \"x\", compression=zipfile.ZIP_DEFLATED) as archive:\n            archive.write(sql_path, arcname=\"results/\" + sql_path.name)\n            archive.write(records_path, arcname=\"records/\" + records_path.name)\n        with zipfile.ZipFile(bundle) as archive:\n            assert archive.testzip() is None\n            assert len(archive.namelist()) == 2\n        report = {\"state\": \"COMPLETED\", \"best_epoch\": context[\"training_report\"][\"best_epoch\"],\n                  \"training_updates\": 0, \"test_examples\": 432, \"test_order_verified\": True,\n                  \"dev_metrics\": dev_summary, \"test_f1\": None,\n                  \"test_f1_note\": \"Obtain from Gradescope; no test labels are provided locally.\",\n                  \"test_sql_execution_errors\": sum(bool(e) for e in errors),\n                  \"sql_file\": str(sql_path), \"records_file\": str(records_path),\n                  \"submission_zip\": str(bundle), \"q6_evidence\": str(output / \"q6_dev_analysis\"),\n                  \"selected_checkpoint\": str(checkpoint),\n                  \"checkpoint_sha256\": sha256(checkpoint / \"model.safetensors\"),\n                  \"test_input_sha256\": sha256(work / \"data/test.nl\"),\n                  \"file_sha256\": {str(p.relative_to(output)): sha256(p) for p in [sql_path, records_path, bundle]}}\n        (output / \"export_report.json\").write_text(json.dumps(report, indent=2))\n        status(\"COMPLETED\", \"q6_evidence_and_q7_files_exported\", test_examples=432)\n        print(\"\\nQ6 EVIDENCE AND Q7 TEST FILES READY\", flush=True)\n        print(json.dumps(report, indent=2), flush=True)\n        print(\"Save this notebook and ask for file checks before submitting to Gradescope.\")\n        print(\"Do not rerun generation to tune against the test set. No Gradescope submission was made automatically.\")\n    except BaseException:\n        status(\"FAILED\", \"export_failed\", error=traceback.format_exc())\n        raise\n\n\nif __name__ == \"__main__\":\n    main(sys.argv[1])\n"
FINAL_SQL_SOURCE = "\"\"\"Runtime-only replacement for compute_records: same records/metrics, killable SQL worker.\"\"\"\nimport os\nfrom pathlib import Path\nimport pickle\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\nWORKER = r'''\nimport concurrent.futures, pickle, sqlite3, sys, time\nfrom pathlib import Path\nqueries_path, results_path, database, budget = sys.argv[1:]\ndeadline = time.monotonic() + float(budget)\nwith open(queries_path, \"rb\") as handle:\n    queries = pickle.load(handle)\ndef execute(index, query):\n    connection = sqlite3.connect(Path(database).resolve().as_uri() + \"?mode=ro\", uri=True)\n    connection.set_progress_handler(lambda: int(time.monotonic() >= deadline), 10000)\n    try:\n        if time.monotonic() >= deadline:\n            return index, [], \"Query timed out\"\n        rows = connection.execute(query).fetchall()\n        return index, rows, \"\"\n    except Exception as error:\n        if time.monotonic() >= deadline:\n            return index, [], \"Query timed out\"\n        return index, [], f\"{type(error).__name__}: {error}\"\n    finally:\n        connection.close()\nwith open(results_path, \"wb\") as output:\n    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as pool:\n        futures = [pool.submit(execute, i, query) for i, query in enumerate(queries)]\n        for future in concurrent.futures.as_completed(futures):\n            pickle.dump(future.result(), output, protocol=pickle.HIGHEST_PROTOCOL)\n            output.flush()\n'''\n\n\ndef compute_records_bounded(queries, database, timeout_secs=120):\n    database = Path(database).resolve()\n    if not database.is_file():\n        raise FileNotFoundError(database)\n    records, errors = [[] for _ in queries], [\"Query timed out\" for _ in queries]\n    directory = \"/content\" if Path(\"/content\").is_dir() else None\n    with tempfile.TemporaryDirectory(prefix=\"hw4_sql_worker_\", dir=directory) as tmp:\n        tmp = Path(tmp)\n        script, incoming, outgoing = tmp / \"worker.py\", tmp / \"queries.pkl\", tmp / \"results.pkl\"\n        script.write_text(WORKER)\n        with incoming.open(\"wb\") as handle:\n            pickle.dump(list(queries), handle, protocol=pickle.HIGHEST_PROTOCOL)\n        print(f\"Executing {len(queries)} SQL queries: {timeout_secs}s batch limit, isolated worker.\", flush=True)\n        started = time.monotonic()\n        process = subprocess.Popen([sys.executable, str(script), str(incoming), str(outgoing), str(database), str(timeout_secs)],\n                                   stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)\n        timed_out = False\n        try:\n            while True:\n                remaining = timeout_secs - (time.monotonic() - started)\n                if remaining <= 0:\n                    timed_out = True\n                    break\n                try:\n                    process.wait(timeout=min(15, remaining))\n                    break\n                except subprocess.TimeoutExpired:\n                    print(f\"SQL worker: {time.monotonic()-started:.0f}s elapsed; still within {timeout_secs}s limit.\", flush=True)\n            if timed_out and process.poll() is None:\n                process.terminate()\n                try:\n                    process.wait(timeout=5)\n                except subprocess.TimeoutExpired:\n                    process.kill()\n                    process.wait(timeout=5)\n            stderr = process.stderr.read()\n            if not timed_out and process.returncode:\n                raise RuntimeError(f\"SQL worker failed: {stderr[-2000:]}\")\n            completed = 0\n            if outgoing.is_file():\n                with outgoing.open(\"rb\") as handle:\n                    while True:\n                        try:\n                            index, rows, error = pickle.load(handle)\n                        except (EOFError, pickle.UnpicklingError):\n                            break  # A killed worker may leave one incomplete trailing record.\n                        assert 0 <= index < len(queries)\n                        records[index], errors[index] = rows, error\n                        completed += 1\n            print(f\"SQL execution finished: {completed}/{len(queries)} completed; \"\n                  f\"{len(queries)-completed} timed out. Worker exited.\", flush=True)\n        finally:\n            if process.poll() is None:\n                process.kill()\n                process.wait(timeout=5)\n            process.stderr.close()\n    return records, errors\n\n\ndef install_bounded_sql(utils_module):\n    # Imported metric/save helpers resolve compute_records through this module's globals.\n    # This changes only this child process, never part2/utils.py on disk.\n    def bounded(queries):\n        return compute_records_bounded(queries, utils_module.DB_PATH, 120)\n    utils_module.compute_records = bounded\n"

# Notebook embeds FINAL_EXPORT_SOURCE and FINAL_SQL_SOURCE before this code.
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
import json
import shutil
import subprocess
import sys

assert sys.platform == "linux" and Path("/content").is_dir(), "Use the existing L4 Colab runtime."
drive.mount("/content/drive")
TRAIN_RUN = Path("/content/drive/MyDrive/NYU_NLP_HW4/part2/continuation_runs/20261110T140531341373Z")
TRAIN_REPORT = json.loads((TRAIN_RUN / "final_report.json").read_text())
assert TRAIN_REPORT["state"] == "COMPLETED" and TRAIN_REPORT["best_epoch"] == 17
FINAL_WORK = Path("/content") / f"hw4_t5_continue_{TRAIN_RUN.name}"
PYTHON2 = "/content/hw4-part2-env/bin/python"
assert FINAL_WORK.is_dir() and Path(PYTHON2).is_file(), (
    "This local runtime was replaced. Stop here and ask to restore the epoch 17 checkpoint first."
)
settings = json.loads((TRAIN_RUN / "configuration.json").read_text())
checkpoint = FINAL_WORK / "checkpoints/ft_experiments" / settings["experiment_name"] / "best"
assert json.loads((checkpoint / "checkpoint_meta.json").read_text())["epoch"] == 17
assert (checkpoint / "model.safetensors").is_file()
for existing in (TRAIN_RUN.parent.parent / "final_exports").glob("*/export_report.json"):
    previous = json.loads(existing.read_text())
    if previous.get("state") == "COMPLETED" and previous.get("selected_checkpoint") == str(checkpoint):
        raise RuntimeError(f"Final test files already exist at {existing.parent}. Inspect those files; do not regenerate.")
subprocess.run([PYTHON2, "-c", "import torch; assert torch.cuda.is_available(); print('GPU:',torch.cuda.get_device_name(0))"], check=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
FINAL_EXPORT = TRAIN_RUN.parent.parent / "final_exports" / stamp
FINAL_EXPORT.mkdir(parents=True, exist_ok=False)
FINAL_HELPERS = FINAL_WORK / "final_export_helpers" / stamp
FINAL_HELPERS.mkdir(parents=True, exist_ok=False)
FINAL_SCRIPT = FINAL_HELPERS / "export_dev_evidence_and_test.py"
FINAL_SCRIPT.write_text(FINAL_EXPORT_SOURCE)
sql_helper = FINAL_HELPERS / "final_bounded_sql.py"
sql_helper.write_text(FINAL_SQL_SOURCE)
for path in [FINAL_SCRIPT, sql_helper]:
    shutil.copy2(path, FINAL_EXPORT / path.name)
FINAL_CONTEXT = FINAL_EXPORT / "export_context.json"
FINAL_CONTEXT.write_text(json.dumps({
    "work_directory": str(FINAL_WORK), "export_directory": str(FINAL_EXPORT),
    "training_run": str(TRAIN_RUN), "training_report": TRAIN_REPORT,
    "configuration": settings, "best_checkpoint_directory": str(checkpoint),
}, indent=2))
(FINAL_EXPORT / "status.json").write_text(json.dumps({"state":"PREPARED_NOT_STARTED",
    "best_epoch":17,"export_directory":str(FINAL_EXPORT)}, indent=2))
print("FIXED MODEL: epoch 17; dev Record F1 87.25%. No model changes.")
print("EXPORT FOLDER:", FINAL_EXPORT)
print("Preparation complete. Test generation has NOT started.")
print("Run Section 2 once to export Q6 evidence and generate the two Q7 files.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FIXED MODEL: epoch 17; dev Record F1 87.25%. No model changes.
EXPORT FOLDER: /content/drive/MyDrive/NYU_NLP_HW4/part2/final_exports/20261110T162054327620Z
Preparation complete. Test generation has NOT started.
Run Section 2 once to export Q6 evidence and generate the two Q7 files.


## 2. Export dev evidence and generate Q7 files — run ONCE

This starts no training. It prints Q6 evidence, generates the 432 test predictions, executes SQL with a 120-second batch budget, checks output counts/order, and creates the submission ZIP. It can take several minutes.


In [2]:
# Run once: export dev evidence and generate the final test files. No training.
import os
import signal
import subprocess
import json
from datetime import datetime, timezone

assert "FINAL_CONTEXT" in globals() and FINAL_CONTEXT.is_file(), "Run Section 1 first."
assert json.loads((FINAL_EXPORT / "status.json").read_text())["state"] == "PREPARED_NOT_STARTED", "Already started; inspect existing results instead of regenerating."
with (FINAL_EXPORT / "GENERATION_STARTED.txt").open("x") as marker:
    marker.write(datetime.now(timezone.utc).isoformat())
print("EXPORT STARTING. Drive output:", FINAL_EXPORT, flush=True)
process = None
try:
    with (FINAL_EXPORT / "export.log").open("w", buffering=1) as log:
        process = subprocess.Popen([PYTHON2, "-u", str(FINAL_SCRIPT), str(FINAL_CONTEXT)],
            cwd=FINAL_WORK, start_new_session=True, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        code = process.wait()
        if code:
            raise RuntimeError(f"Export exited with code {code}. Preserve output and ask for review.")
except BaseException:
    if process is not None and process.poll() is None:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait(timeout=15)
    report_file = FINAL_EXPORT / "export_report.json"
    finished = report_file.is_file() and json.loads(report_file.read_text()).get("state") == "COMPLETED"
    (FINAL_EXPORT / "status.json").write_text(json.dumps({
        "state": "COMPLETED" if finished else "FAILED_OR_INTERRUPTED",
        "stage": "log_reader_interrupted" if finished else "export_interrupted_or_failed",
        "export_directory": str(FINAL_EXPORT),
    }, indent=2))
    print("Existing export files/logs remain at:", FINAL_EXPORT)
    raise
else:
    assert json.loads((FINAL_EXPORT / "status.json").read_text())["state"] == "COMPLETED"
    print("\nSAVE THIS NOTEBOOK AND ASK FOR REVIEW BEFORE SUBMITTING.")
    print("Drive output:", FINAL_EXPORT)


EXPORT STARTING. Drive output: /content/drive/MyDrive/NYU_NLP_HW4/part2/final_exports/20261110T162054327620Z
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Q6 DEV EVIDENCE EXPORTED
{
  "model_epoch": 17,
  "dev_examples": 466,
  "record_f1": 0.8724690312780573,
  "record_em": 0.8540772532188842,
  "sql_em": 0.723175965665236,
  "sql_execution_errors": 30,
  "record_mismatches": 68,
  "sql_text_mismatches": 129,
  "review_group_counts_out_of_466": {
    "SQL syntax/format error": 19,
    "Valid SQL but incorrect records: manual semantic review needed": 46,
    "SQL identifier/reference error": 9,
    "SQL execution timeout": 2
  },
  "note": "Execution groups are measured from saved error messages. Semantic hints are NOT final error classifications; inspect examples before writing Q6."
}
Q6 EXAMPLES FOR REVIEW (not a finished written answer):
[
  {
    "review_group": "SQL syntax/format error",
    "example_number_1_based":

## 3. Inspect saved files — read-only

Use after completion or reconnecting. Do not rerun Section 2 to view results.


In [4]:
# Read-only: inspect saved results without regenerating test predictions.
from pathlib import Path
from google.colab import drive
import json

drive.mount("/content/drive")
if "FINAL_EXPORT" not in globals():
    folders = sorted(Path("/content/drive/MyDrive/NYU_NLP_HW4/part2/final_exports").glob("*"))
    assert folders, "No final export folders found."
    FINAL_EXPORT = folders[-1]
print("EXPORT FOLDER:", FINAL_EXPORT)
for name in ["status.json", "export_report.json", "q6_dev_analysis/summary.json"]:
    path = FINAL_EXPORT / name
    if path.is_file():
        print("\n" + name + ":\n" + path.read_text())
for name in ["results/t5_ft_experiment_test.sql", "records/t5_ft_experiment_test.pkl", "Q7_submission_files.zip"]:
    path = FINAL_EXPORT / name
    print(name, f"{path.stat().st_size:,} bytes" if path.is_file() else "not yet created")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
EXPORT FOLDER: /content/drive/MyDrive/NYU_NLP_HW4/part2/final_exports/20261110T162054327620Z

status.json:
{
  "state": "COMPLETED",
  "stage": "q6_evidence_and_q7_files_exported",
  "updated_utc": "2026-11-10T16:23:53.801460+00:00",
  "best_epoch": 17,
  "export_directory": "/content/drive/MyDrive/NYU_NLP_HW4/part2/final_exports/20261110T162054327620Z",
  "training_updates": 0,
  "test_examples": 432
}

export_report.json:
{
  "state": "COMPLETED",
  "best_epoch": 17,
  "training_updates": 0,
  "test_examples": 432,
  "test_order_verified": true,
  "dev_metrics": {
    "model_epoch": 17,
    "dev_examples": 466,
    "record_f1": 0.8724690312780573,
    "record_em": 0.8540772532188842,
    "sql_em": 0.723175965665236,
    "sql_execution_errors": 30,
    "record_mismatches": 68,
    "sql_text_mismatches": 129,
    "review_group_counts_out_of_466": {
      "SQL